# OOD Test Preparations
Two OOD test sets for a model trained on ExioNAICS. Both exposed via a `text` column.

In [41]:
import pandas as pd
import re

## 1. OOD Set 1 — classification-dataset-v1.csv

In [42]:
clf = pd.read_csv('classification-dataset-v1.csv')
print(clf.shape)
clf.head(2)

(73974, 10)


,Category,website,company_name,homepage_text,h1,h2,h3,nav_link_text,meta_keywords,meta_description
0,Commercial Services & Supplies,bipelectric.com,bip dipietro electric inc,Electrici...,NaN,NaN,NaN,NaN,"electricians vero beach, vero beach electrical...","Providing quality, reliable full service resid..."
1,Healthcare,eliasmedical.com,elias medical,site map | en español Elias Medical h...,Offering Bakersfield family medical care from ...,Welcome to ELIAS MEDICAL#sep#Family Medical Pr...,Get To Know Elias Medical#sep#Family Medical P...,NaN,Elias Medical bakersfield ca family doctor med...,For the best value in Bakersfield skin care tr...


In [43]:
clf_ood = clf[['company_name', 'meta_description', 'Category']].copy()
clf_ood = clf_ood.rename(columns={'Category': 'label', 'meta_description': 'text'})
clf_ood = clf_ood.dropna(subset=['label', 'text']).reset_index(drop=True)

print(clf_ood.shape)
print(clf_ood['label'].value_counts())
clf_ood.head(3)

(66886, 3)
label
Professional Services             6655
Healthcare                        6534
Corporate Services                6442
Financials                        6278
Commercial Services & Supplies    5856
Media, Marketing & Sales          5798
Transportation & Logistics        5727
Information Technology            5426
Energy & Utilities                5162
Consumer Staples                  4906
Industrials                       3073
Consumer Discretionary            2611
Materials                         2418
Name: count, dtype: int64


,company_name,text,label
0,bip dipietro electric inc,"Providing quality, reliable full service resid...",Commercial Services & Supplies
1,elias medical,For the best value in Bakersfield skin care tr...,Healthcare
2,koops overhead doors,"Koops Overhead Doors specializes in the sales,...",Commercial Services & Supplies


## 2. OOD Set 2 — wikidata_global_companies_info.csv

In [44]:
wiki = pd.read_csv('wikidata_global_companies_info.csv', index_col=0)
print(wiki.shape)
wiki.head(2)

(3579, 9)


,wikidata_uri,company name,description,country,instance of,inception,official website,industry,founded by
0,http://www.wikidata.org/entity/Q279260,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",Austria,['hotel' 'Viennese coffee house'],1876-01-01,http://www.sacher.com/,['hotel'],Édouard Sacher
1,http://www.wikidata.org/entity/Q193326,Goldman Sachs,American investment bank,United States of America,['stock exchange' 'bank' 'multinational corpor...,1869-01-01,https://www.goldmansachs.com/,['financial services' 'financial sector'\n 'fi...,Samuel Sachs


In [45]:
def parse_industry(s):
    if pd.isna(s): return None
    s = re.sub(r"[\[\]']", '', str(s))
    return re.sub(r'\s+', ' ', s).strip() or None

wiki['industry_clean'] = wiki['industry'].apply(parse_industry)

wiki_ood = wiki[['company name', 'description', 'industry_clean']].copy()
wiki_ood = wiki_ood.rename(columns={'company name': 'company_name', 'description': 'text', 'industry_clean': 'label'})
wiki_ood = wiki_ood.dropna(subset=['label', 'text']).reset_index(drop=True)

print(wiki_ood.shape)
print(wiki_ood['label'].value_counts().head(10))
wiki_ood.head(3)

(3579, 3)
label
retail                 190
telecommunications     109
automotive industry    108
financial services      98
film industry           65
petroleum industry      63
filmmaking              54
video game industry     51
creative industries     49
software industry       47
Name: count, dtype: int64


,company_name,text,label
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",hotel
1,Goldman Sachs,American investment bank,financial services financial sector financial ...
2,Sberbank,Russian banking and financial services company,banking in Russia financial sector financial s...


## 3. Summary

In [46]:
print("=== OOD Set 1 (Classification Dataset) ===")
print(f"  Rows     : {len(clf_ood)}")
print(f"  Labels   : {clf_ood['label'].nunique()} industry categories")
print(f"  Text len : mean={clf_ood['text'].str.len().mean():.0f} chars")

print("\n=== OOD Set 2 (Wikidata) ===")
print(f"  Rows     : {len(wiki_ood)}")
print(f"  Labels   : unlabeled")
print(f"  Text len : mean={wiki_ood['text'].str.len().mean():.0f} chars")

=== OOD Set 1 (Classification Dataset) ===
  Rows     : 66886
  Labels   : 13 industry categories
  Text len : mean=196 chars

=== OOD Set 2 (Wikidata) ===
  Rows     : 3579
  Labels   : unlabeled
  Text len : mean=35 chars
